# Textile Defect Detection in Google Colab

This notebook tests the trained YOLO model and returns defect coordinates. It supports loading files from Google Drive or uploading them directly.

## 1. Install Required Libraries

Install the model and image-processing dependencies in the Colab runtime.

In [ ]:
%pip install -q ultralytics pillow matplotlib fastapi python-multipart httpx
print("Required libraries installed successfully")

## 2. Mount Google Drive

Run this cell and authorize access when Colab prompts you.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
print(f'Drive mounted: {DRIVE_ROOT.exists()}')

## 3. Upload or Access Project Files

If the model and image are already in Google Drive, set `DRIVE_PROJECT_DIR`. Otherwise, leave it as `None` and upload both files when prompted.

In [ ]:
from google.colab import files
import shutil

DRIVE_PROJECT_DIR = None  # Example: DRIVE_ROOT / 'Textile defect detection/Backend_fastapi_endpoint'
WORK_DIR = Path('/content/textile_defect_detection')
WORK_DIR.mkdir(exist_ok=True)

if DRIVE_PROJECT_DIR is not None:
    DRIVE_PROJECT_DIR = Path(DRIVE_PROJECT_DIR)
    model_source = DRIVE_PROJECT_DIR / 'best.pt'
    image_candidates = list(DRIVE_PROJECT_DIR.glob('*.jpg')) + list(DRIVE_PROJECT_DIR.glob('*.jpeg')) + list(DRIVE_PROJECT_DIR.glob('*.png'))
    if not model_source.exists() or not image_candidates:
        raise FileNotFoundError('Drive folder must contain best.pt and at least one JPG, JPEG, or PNG image.')
    shutil.copy2(model_source, WORK_DIR / 'best.pt')
    shutil.copy2(image_candidates[0], WORK_DIR / image_candidates[0].name)
else:
    print('Upload best.pt and one test image, such as hole.jpeg')
    uploaded = files.upload()
    for filename in uploaded:
        shutil.move(filename, WORK_DIR / filename)

model_path = WORK_DIR / 'best.pt'
image_files = [path for path in WORK_DIR.iterdir() if path.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
assert model_path.exists(), 'best.pt was not found'
assert image_files, 'No test image was found'
image_path = image_files[0]
print(f'Model: {model_path}')
print(f'Test image: {image_path}')

## 4. Set Working Directory

In [ ]:
import os

os.chdir(WORK_DIR)
print(f'Working directory: {Path.cwd()}')
print('Files:', [path.name for path in Path.cwd().iterdir()])

## 5. Run the Target Code in Colab

Load the trained YOLO model and run inference on the selected image.

In [ ]:
import json
from ultralytics import YOLO

model = YOLO(str(model_path))
results = model.predict(source=str(image_path), verbose=False)
result = results[0]

detections = []
if result.boxes is not None:
    for box, confidence, class_id in zip(
        result.boxes.xyxy.cpu().tolist(),
        result.boxes.conf.cpu().tolist(),
        result.boxes.cls.cpu().tolist(),
    ):
        class_id = int(class_id)
        detections.append({
            'class_id': class_id,
            'class_name': result.names.get(class_id, str(class_id)),
            'confidence': round(float(confidence), 6),
            'coordinates': {
                'x1': round(float(box[0]), 2),
                'y1': round(float(box[1]), 2),
                'x2': round(float(box[2]), 2),
                'y2': round(float(box[3]), 2),
            },
        })

output = {
    'filename': image_path.name,
    'image_width': result.orig_shape[1],
    'image_height': result.orig_shape[0],
    'detections': detections,
}
print(json.dumps(output, indent=2))

### Test the FastAPI `/predict` Contract

This uses FastAPI's in-process test client, so no public server or tunneling is needed.

In [ ]:
from fastapi import FastAPI, File, UploadFile
from fastapi.testclient import TestClient

api_app = FastAPI()

@api_app.post('/predict')
async def predict(file: UploadFile = File(...)):
    api_result = model.predict(source=str(image_path), verbose=False)[0]
    api_detections = []
    if api_result.boxes is not None:
        for box, confidence, class_id in zip(
            api_result.boxes.xyxy.cpu().tolist(),
            api_result.boxes.conf.cpu().tolist(),
            api_result.boxes.cls.cpu().tolist(),
        ):
            class_id = int(class_id)
            api_detections.append({
                'class_id': class_id,
                'class_name': api_result.names.get(class_id, str(class_id)),
                'confidence': float(confidence),
                'coordinates': {
                    'x1': float(box[0]), 'y1': float(box[1]),
                    'x2': float(box[2]), 'y2': float(box[3]),
                },
            })
    return {'filename': file.filename, 'detections': api_detections}

with image_path.open('rb') as image_file:
    response = TestClient(api_app).post(
        '/predict',
        files={'file': (image_path.name, image_file, 'image/jpeg')},
    )

assert response.status_code == 200
assert 'detections' in response.json()
print('FastAPI /predict test passed')
print(json.dumps(response.json(), indent=2))

## 6. Validate Outputs

Display the annotated image and verify that every detection contains the expected fields.

In [ ]:
from IPython.display import Image, display

required_keys = {'class_id', 'class_name', 'confidence', 'coordinates'}
for detection in detections:
    assert required_keys.issubset(detection)
    assert set(detection['coordinates']) == {'x1', 'y1', 'x2', 'y2'}

annotated_path = WORK_DIR / 'annotated_result.jpg'
result.save(filename=str(annotated_path))
display(Image(filename=str(annotated_path)))
print(f'Validation passed: {len(detections)} detection(s) returned')